# Summarize convention results

Consolidates the two-convention SrTiO3 Born-charge calculations (`.mmn`, centre-aware, pair-dependent H/A vs. direct `_r.dat` + `_hr.dat`) across three Wannier trials into one comparison table and plot.

**Note:** `BORN_RESULTS` below points at `calculations/SrTiO3/born_effective_charge/`, a centralized results tree. `born_effective_charge.ipynb`'s own docstring says results now live next to each run instead (`<mode_w90>/born_charge/`), and no `calculations/SrTiO3/born_effective_charge/` directory exists in this checkout -- so this script's source paths look stale relative to the current layout. That is a pre-existing issue, not something introduced by this notebook conversion; the paths are preserved as-is. Point `BORN_RESULTS` at wherever your per-trial NPZ files actually live before running.

Set `RUN_COMPUTATION = False` to skip straight to loading a previously saved `born_charge_comparison.csv` / plot from `OUTPUT`.

In [ ]:
import sys
import csv
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "validation").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from validation._paths import RESULTS, STO

## Config

In [ ]:
RUN_COMPUTATION = False  # True: reread the per-trial NPZ files and rebuild the table+plot. False: load a previously saved table/plot from OUTPUT.

BORN_RESULTS = STO / "born_effective_charge"
OUTPUT = BORN_RESULTS / "convention_symmetry"

TRIALS = {
    "a": "trial_01_Sr_spd_Ti_spd_O_sp",
    "b": "trial_02_Sr_p_O_sp",
    "c": "trial_03_Sr_sp_Ti_pd_O_sp",
}
FILES = {
    "mmn_pair": "born_charge_Ti_Q_01A_mmn_taucentres_mhn0_wbnone.npz",
    "rdat_direct": "born_charge_Ti_Q_01A_mhn0_nows.npz",
}
STYLES = {
    "mmn_pair": dict(label=".mmn, centre-aware; pair-dependent H/A", color="C0", marker="o"),
    "rdat_direct": dict(label="direct _r.dat + _hr.dat", color="C1", marker="s"),
}

## Load each trial/convention NPZ

`results[(trial, convention)]` after this cell holds the raw arrays (`z_int`, `z_cross`, `z_ext`, `z_total`, `nks`, `z_qe`, ...) for every (trial, convention) pair -- inspect directly.

In [ ]:
if RUN_COMPUTATION:
    expected_nks = np.array([4, 6, 8, 10, 12, 14, 16])
    results = {}
    for trial, directory in TRIALS.items():
        for convention, filename in FILES.items():
            path = BORN_RESULTS / directory / filename
            with np.load(path, allow_pickle=False) as data:
                row = {key: data[key] for key in data.files}
            if not np.array_equal(row["nks"], expected_nks):
                raise ValueError(f"{path} does not contain the full requested nk sweep")
            source = str(row["position_source"][0])
            wanted = "mmn" if convention == "mmn_pair" else "rdat"
            if source != wanted or str(row["embedding"][0]) != "centres":
                raise ValueError(f"{path} has unexpected convention metadata")
            if "min_hopping_norm" in row and float(row["min_hopping_norm"]) != 0.0:
                raise ValueError(f"{path} did not retain every hopping")
            if "ws_reassign" in row:
                expected_ws = convention == "mmn_pair"
                if bool(row["ws_reassign"]) != expected_ws:
                    raise ValueError(f"{path} has an inconsistent H(R) convention")
            results[(trial, convention)] = row

## Build the comparison table

In [ ]:
if RUN_COMPUTATION:
    OUTPUT.mkdir(parents=True, exist_ok=True)
    table_path = OUTPUT / "born_charge_comparison.csv"
    with table_path.open("w", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(["trial", "convention", "nk", "z_qe_zz", "z_int_zz", "z_cross_zz", "z_ext_zz",
                         "z_total_zx", "z_total_zy", "z_total_zz", "zz_minus_qe", "zz_relative_percent"])
        for trial in TRIALS:
            for convention in FILES:
                data = results[(trial, convention)]
                qe = float(data["z_qe"][2])
                for i, nk in enumerate(data["nks"]):
                    total = np.real(data["z_total"][i])
                    resid = float(total[2] - qe)
                    writer.writerow([
                        trial, convention, int(nk), f"{qe:.8f}",
                        f"{np.real(data['z_int'][i, 2]):.8f}", f"{np.real(data['z_cross'][i, 2]):.8f}",
                        f"{np.real(data['z_ext'][i, 2]):.8f}", f"{total[0]:.8e}", f"{total[1]:.8e}",
                        f"{total[2]:.8f}", f"{resid:.8f}", f"{100.0 * resid / qe:.6f}",
                    ])
    print(f"saved {table_path}")

## Plot

In [ ]:
def make_plot(results):
    fig, axes = plt.subplots(2, 3, figsize=(13.0, 7.2), dpi=150, sharex="col")
    for col, trial in enumerate(TRIALS):
        ax_z = axes[0, col]
        ax_zero = axes[1, col]
        qe = float(results[(trial, "mmn_pair")]["z_qe"][2])
        ax_z.axhline(qe, color="0.25", linewidth=0.9, label="DFPT")
        for convention, style in STYLES.items():
            data = results[(trial, convention)]
            nks = data["nks"]
            total = np.real(data["z_total"])
            ax_z.plot(nks, total[:, 2], color=style["color"], marker=style["marker"],
                     linewidth=1.4, markersize=3.8, label=style["label"])
            forbidden = np.maximum(np.abs(total[:, 0]), np.abs(total[:, 1]))
            ax_zero.semilogy(nks, np.maximum(forbidden, 1e-18), color=style["color"],
                            marker=style["marker"], linewidth=1.4, markersize=3.8)
        ax_z.set_title(f"set ({trial})")
        ax_z.grid(alpha=0.18)
        ax_zero.grid(alpha=0.18)
        ax_zero.set_xlabel("interpolation mesh $n_k$")
        if col == 0:
            ax_z.set_ylabel("$Z^*_{Ti,zz}$")
            ax_zero.set_ylabel("max$(|Z^*_{Ti,zx}|,|Z^*_{Ti,zy}|)$")
    axes[0, 0].legend(frameon=False, fontsize=8)
    fig.suptitle("SrTiO$_3$ Born charge: consistent-convention comparison")
    fig.tight_layout()
    return fig

In [ ]:
if RUN_COMPUTATION:
    fig = make_plot(results)
    plot_path = OUTPUT / "born_charge_convention_comparison.png"
    fig.savefig(plot_path, bbox_inches="tight")
    print(f"saved {plot_path}")
    plt.show()

## Load a previous run instead (skip if you just computed above)

In [ ]:
if not RUN_COMPUTATION:
    table_path = OUTPUT / "born_charge_comparison.csv"
    if table_path.exists():
        with table_path.open() as f:
            saved_rows = list(csv.DictReader(f))
        for row in saved_rows:
            print(row)
        plot_path = OUTPUT / "born_charge_convention_comparison.png"
        if plot_path.exists():
            from IPython.display import Image, display
            display(Image(filename=str(plot_path)))
    else:
        print(f"no saved table at {table_path} -- set RUN_COMPUTATION = True, "
              "or point BORN_RESULTS/OUTPUT at wherever your per-trial results live.")